# 05b — Silver RENAMU: tabla `renamu_finanzas`

Construye la tabla Silver del RENAMU con el mapeo de `data/silver/renamu_mapeo.yaml` (definido y verificado en `05a`).

**Flujo:** primero se cargan todas las filas tal cual. Luego se limpia paso a paso, mostrando qué cambia en cada paso. Al final se aplican las reglas de calidad.

| Parte | Qué hace |
| --- | --- |
| 0 | Configuración |
| 1 | Carga cruda: todas las filas de Bronze, solo las columnas del mapeo, como texto |
| 2 | Diagnóstico inicial (antes de limpiar) |
| 3 | Limpieza paso a paso, con el conteo de cambios de cada paso |
| 4 | Comparación antes / después y registro de cambios |
| 5 | Reglas de calidad: todas en una celda, con un resumen; 5.1 cuadre |
| 6 | Escritura en Parquet (Silver y cuarentena) y verificación |
| 7 | Bitácora (formato de auditoría) y `data/auditoria/volumenes.parquet` |
| 8 | Resumen final |

**Grano:** una fila por municipalidad (`ubigeo`) y edición del RENAMU.

## 0. Configuración

In [1]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ARQUITECTURA.md").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import uuid
from datetime import datetime, timezone
from functools import reduce
import pandas as pd
from pyspark.sql import functions as F, Window

from src.config import cargar_config, cargar_reglas, ruta
from src.spark import crear_sesion
from src import bronze
from src.perfilado import detectar_formato, leer_csv, a_pandas
from src.renamu import cargar_mapeo
from src.silver import aplicar          # paso de limpieza: cuenta cambios y muestra ejemplos

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
cfg = cargar_config()
REGLAS = {r["id"]: r for r in cargar_reglas()}
spark = crear_sesion("silver_renamu")
spark.sparkContext.setLogLevel("ERROR")   # oculta los avisos de rendimiento de Spark
CAMBIOS = []                              # registro de cambios de todos los pasos

inicio = datetime.now(timezone.utc)
RUN_ID = f"silver_renamu_{inicio:%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:6]}"

MAPEO = cargar_mapeo(ruta("silver") / "renamu_mapeo.yaml")
ID = MAPEO["identificacion"]
ED = {a: e for a, e in MAPEO["ediciones"].items() if e.get("existe", True) is not False}
ING = [f"ing_{v}" for v in MAPEO["variables"]["ingresos"]]
GAS = [f"gas_{v}" for v in MAPEO["variables"]["gastos"]]
MONTOS = [c for c in ING + GAS if not c.endswith("_informo")]
BANDERAS = ["ing_informo", "gas_informo"]
CODIGOS = ["ccdd", "ccpp", "ccdi"]

man = pd.DataFrame(bronze.leer_manifest()["archivos"])
f3 = man[(man["fuente"] == "F3") & (man["origen"] == "extraido_de_zip")].copy()
f3 = f3[~f3["ruta"].str.contains("_F02_")]
f3["anio"] = f3["anio"].astype(int)
print("run_id:", RUN_ID)
print("Ediciones con cuadro:", sorted(ED), "| sin cuadro:", sorted(set(MAPEO["ediciones"]) - set(ED)))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/22 14:47:52 WARN Utils: Your hostname, Rodrigo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/22 14:47:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 14:47:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


run_id: silver_renamu_20261122T194755Z_ad7628
Ediciones con cuadro: [2009, 2010, 2011, 2012, 2013, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026] | sin cuadro: [2014, 2016]


## 1. Carga cruda

- **Qué se lee:** todas las filas de cada archivo del mapeo, todo como texto, sin limpiar. Solo se leen:
  - la identificación: `ccdd`, `ccpp`, `ccdi` y el tipo de municipalidad. Los nombres de departamento, provincia y distrito **no se cargan**: la conexión es el `ubigeo` y en Gold los nombres se toman de una sola fuente (SIAF);
  - las columnas del cuadro de ingresos o gastos.

In [2]:
ARCHIVOS = {}                                   # (edicion, archivo) -> bloques que trae
for a, e in ED.items():
    for b in ("ingresos", "gastos"):
        ARCHIVOS.setdefault((a, e[b]["archivo"]), []).append(b)

def cargar(a, archivo, bloques):
    r = f3[(f3["anio"] == a) & (f3["archivo"] == archivo)]
    assert len(r) == 1, f"{a} {archivo}: {len(r)} coincidencias en el manifest"
    r = r.iloc[0]
    p = RAIZ / r["ruta"]
    df = leer_csv(spark, str(p), detectar_formato(p))
    real = {c.upper(): c for c in df.columns}
    def primera(lista):
        c = next((real[x] for x in lista if x in real), None)
        return F.col(c) if c else F.lit(None).cast("string")
    sel = [F.lit(a).alias("edicion"), F.lit(archivo).alias("archivo"), F.lit(",".join(bloques)).alias("bloques"),
           *[F.col(real[c]).alias(c.lower()) for c in ID["ubigeo"]],
           primera(ID["tipo_municipalidad"]).alias("tipo")]
    for b, pref in (("ingresos", "ing_"), ("gastos", "gas_")):
        cods = {k: v for k, v in ED[a][b].items() if k != "archivo"}
        for v, c in cods.items():
            if b not in bloques or c is None:
                sel.append(F.lit(None).cast("string").alias(pref + v))
            elif isinstance(c, list):            # lista = sumar esas columnas (paso 3.8)
                sel.append(F.col(real[c[0]]).alias(pref + v))
                sel += [F.col(real[x]).alias(f"{pref}{v}__suma{k}") for k, x in enumerate(c[1:], 1)]
            else:
                sel.append(F.col(real[c]).alias(pref + v))
    orden = Window.orderBy(F.monotonically_increasing_id())
    return (df.select(*sel)
              .withColumn("_fila_origen", F.row_number().over(orden))
              .withColumn("_archivo_origen", F.lit(r["ruta"]))
              .withColumn("_sha256_origen", F.lit(r["sha256"]))
              .withColumn("_fecha_descarga", F.lit(r["fecha_descarga"])))

crudo = reduce(lambda x, y: x.unionByName(y, allowMissingColumns=True), [cargar(a, f, b) for (a, f), b in ARCHIVOS.items()]).localCheckpoint()
foto_inicial = a_pandas(crudo.groupBy("edicion", "archivo", "bloques").count().orderBy("edicion", "archivo"))
print("Filas cargadas:", int(foto_inicial["count"].sum()), "en", len(foto_inicial), "archivos")
display(foto_inicial.rename(columns={"count": "filas"}).set_index(["edicion", "archivo"]))

Filas cargadas: 42747 en 23 archivos


bloques  filas
edicion archivo                                                 
2009    C094.csv                                 ingresos   1834
        C095.csv                                   gastos   1834
2010    C094.csv                                 ingresos   1834
        C095.csv                                   gastos   1834
2011    C094.csv                                 ingresos   1838
        C095.csv                                   gastos   1838
2012    C104.csv                                 ingresos   1838
        C105.csv                                   gastos   1838
2013    C104.csv                                 ingresos   1838
        C105.csv                                   gastos   1838
2015    C111.csv                                 ingresos   1853
        C112.csv                                   gastos   1853
2017    C115.csv                                 ingresos   1871
        C116.csv                                   gastos   1871
2018    C06.csv                           ingresos,gastos   1874
2019    C04.csv                           ingresos,gastos   1874
2020    C04.csv                           ingresos,gastos   1874
2021    Base_Renamu_2021_vf.csv           ingresos,gastos   1874
2022    Base_RENAMU_2022_f.csv            ingresos,gastos   1874
2023    Base-Datos_2023_f.csv             ingresos,gastos   1891
2024    Base-Datos_2024_f.csv             ingresos,gastos   1891
2025    Base-Datos_2025_f_.csv            ingresos,gastos   1891
2026    Base_de_Datos_F01_RENAMU2026.csv  ingresos,gastos   1892

## 2. Diagnóstico inicial (antes de limpiar)

Qué problemas trae cada edición tal como viene de Bronze. En la parte 3 se ve cómo se resuelve cada uno, y las reglas de calidad (parte 5) deciden qué hacer con lo que no se resuelve limpiando. Cada tabla muestra solo las columnas con algún caso.

In [3]:
texto = [c for c in crudo.columns if c not in ("edicion", "archivo", "bloques", "_fila_origen") and not c.startswith("_")]
datos = ["tipo"] + [c for c in texto if c.startswith(("ing_", "gas_"))]
vacio = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == "")
# montos que el mapeo define en cada edición (sin las columnas auxiliares de suma ni las categorías que no existen, p. ej. la III de 2009)
EDS_COL = {pref + v: [a for a, e in ED.items() if e[b].get(v) is not None]
           for b, pref in (("ingresos", "ing_"), ("gastos", "gas_")) for v in MAPEO["variables"][b] if v != "informo"}
monto_cols = list(EDS_COL)
NUM = r"^-?(\d+([.,]\d*)?|[.,]\d+)$"          # 1234 · 1234,5 · 1234.5 · .9 (decimal sin cero inicial)

def contar(cond):
    return F.sum(F.coalesce(cond, F.lit(False)).cast("int"))

def presente(c):
    """La columna pertenece a un bloque que este archivo sí trae (en 2009–2017 ingresos y gastos vienen en archivos distintos)."""
    bloque = F.col("bloques").contains("ingresos" if c.startswith("ing_") else "gastos")
    return bloque & F.col("edicion").isin(EDS_COL[c]) if c in EDS_COL else bloque

ub_crudo = F.concat_ws("|", *[F.trim(F.col(c)) for c in CODIGOS])
w_dup = Window.partitionBy("edicion", "archivo", "ub_crudo")
d = crudo.withColumn("ub_crudo", ub_crudo).withColumn("_rep", F.count("*").over(w_dup))

def mostrar(df):
    """Muestra solo las columnas con algún caso; las que están en 0 en todas las ediciones se nombran aparte."""
    sin = [c for c in df.columns if c != "filas" and (df[c] == 0).all()]
    display(df.drop(columns=sin))
    print("Sin casos en ninguna edición:", ", ".join(sin) or "—")

### 2.1 Identificación (códigos de ubicación → `ubigeo`)

In [4]:
mostrar(a_pandas(d.groupBy("edicion").agg(
    F.count("*").alias("filas"),
    contar(reduce(lambda a, b: a & b, [vacio(c) for c in datos])).alias("filas_totalmente_vacias"),
    contar(reduce(lambda a, b: a | b, [vacio(c) for c in CODIGOS])).alias("codigo_vacio"),
    contar(reduce(lambda a, b: a | b, [~F.trim(F.col(c)).rlike(r"^\d+$") & ~vacio(c) for c in CODIGOS])).alias("codigo_no_numerico"),
    contar(F.trim(F.col("ccdd")).isin("0", "00")).alias("departamento_00"),
    contar(reduce(lambda a, b: a | b, [F.trim(F.col(c)).rlike(r"^\d$") for c in CODIGOS])).alias("codigo_sin_cero_izq"),
    contar(F.col("_rep") > 1).alias("ubigeo_repetido_en_archivo"),
    contar(reduce(lambda a, b: a | b, [F.col(c) != F.trim(F.col(c)) for c in texto])).alias("filas_con_espacios_sobrantes"),
).orderBy("edicion")).set_index("edicion"))

,filas,codigo_sin_cero_izq,filas_con_espacios_sobrantes
edicion,,,
2009,3668,0,32
2010,3668,0,22
2011,3676,0,12
2012,3676,0,0
2013,3676,0,3
2015,3706,1,2
2017,3742,0,0
2018,1874,0,0
2019,1874,0,1


Sin casos en ninguna edición: filas_totalmente_vacias, codigo_vacio, codigo_no_numerico, departamento_00, ubigeo_repetido_en_archivo


**Diagnóstico:** la identificación viene completa. No hay filas vacías, códigos vacíos o no numéricos, departamento `00` ni `ubigeo` repetido. Hay dos detalles de formato, que se corrigen en la limpieza:
- **códigos sin cero a la izquierda**: 920 filas en 2023 (`1` en vez de `01`), más 4 en 2024 y 1 en 2015 → paso 3.5;
- **espacios sobrantes**: 2009–2015, 2019 y 2023 → paso 3.2.

### 2.2 Montos y banderas de informante

Montos en celdas; banderas en filas.

In [5]:
celdas = lambda cond: reduce(lambda a, b: a + b, [F.coalesce(presente(c) & cond(c), F.lit(False)).cast("int") for c in monto_cols])
mostrar(a_pandas(crudo.groupBy("edicion").agg(
    F.sum(celdas(vacio)).alias("montos_vacios"),
    F.sum(celdas(lambda c: F.col(c).contains(","))).alias("con_coma_decimal"),
    F.sum(celdas(lambda c: F.col(c).contains("."))).alias("con_punto_decimal"),
    F.sum(celdas(lambda c: ~vacio(c) & ~F.trim(F.col(c)).rlike(NUM))).alias("no_numericos"),
    F.sum(celdas(lambda c: F.trim(F.col(c)).startswith("-"))).alias("negativos"),
    contar(reduce(lambda a, b: a | b, [presente(c) & vacio(c) for c in BANDERAS])).alias("filas_bandera_vacia"),
).orderBy("edicion")).set_index("edicion"))

,montos_vacios,con_coma_decimal,con_punto_decimal,negativos,filas_bandera_vacia
edicion,,,,,
2009,128,10170,0,0,0
2010,99,12942,0,0,0
2011,54,12490,0,1,0
2012,0,12604,0,0,0
2013,3,12811,0,0,0
2015,0,12657,0,0,0
2017,0,0,0,0,0
2018,0,12936,0,1,0
2019,5,12906,0,0,0


Sin casos en ninguna edición: no_numericos


**Diagnóstico:**
- **Separador decimal:** coma en casi todas las ediciones y **punto en 2023–2024**. 2017 viene en enteros. No hay separador de miles ni valores no numéricos, así que basta con cambiar `,` por `.` (paso 3.7).
- **Montos vacíos:** pocos. En 2009–2011 y 2023 son las municipalidades que no entregaron el cuadro (la regla `SLV-F3-007` las registra). En 2013 (3) y 2019 (5) son celdas sueltas. En todos los casos quedan nulos, no 0.
- **Negativos:** 5 celdas (2011, 2018 y 2025). Se conservan y la regla `SLV-F3-006` los registra. En 2009 aparecen otros 3 al sumar el saldo de balance (`P102_74`, paso 3.8).
- **Bandera vacía:** 1 fila en 2023.

### 2.3 Tipo de municipalidad

Catálogo del INEI: 1 = provincial, 2 = distrital, 3 = centro poblado. Solo se listan las ediciones con algún valor distinto de 1 o 2.

In [6]:
tipos = a_pandas(crudo.groupBy("edicion").agg(
    *[contar(F.trim(F.col("tipo")) == v).alias(f"tipo_{v}") for v in ("1", "2", "3")],
    contar(~F.trim(F.col("tipo")).isin("1", "2", "3") | vacio("tipo")).alias("tipo_otro_o_vacio"),
).orderBy("edicion")).set_index("edicion")
print(f"Tipo 1: {int(tipos['tipo_1'].sum())} filas · tipo 2: {int(tipos['tipo_2'].sum())} filas")
raros = tipos[(tipos["tipo_3"] + tipos["tipo_otro_o_vacio"]) > 0]
display(raros) if len(raros) else print("Todas las filas son tipo 1 o 2.")

Tipo 1: 4497 filas · tipo 2: 38248 filas


,tipo_1,tipo_2,tipo_3,tipo_otro_o_vacio
edicion,,,,
2015,391,3313,0,2


**Diagnóstico:** todas las filas son tipo 1 (provincial) o 2 (distrital); no hay centros poblados. La única excepción son **2 filas de 2015 con el tipo vacío** (en `C112.csv`, gastos). Se resuelve en 3.9, porque la identificación se toma del archivo de ingresos.

## 3. Limpieza paso a paso

### 3.1 Nombres de columnas

Cada código del cuadro pasa a su nombre universal. La tabla muestra el código original de cada edición.

In [7]:
filas = []
for a, e in ED.items():
    for b, pref in (("ingresos", "ing_"), ("gastos", "gas_")):
        for v, c in e[b].items():
            if v != "archivo":
                filas.append({"variable": pref + v, "edicion": a, "codigo": "—" if c is None else " + ".join(c) if isinstance(c, list) else c})
display(pd.DataFrame(filas).pivot(index="variable", columns="edicion", values="codigo").loc[ING + GAS])

edicion,2009,2010,2011,2012,2013,2015,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
variable,,,,,,,,,,,,,,,,
ing_informo,VFI_P102,VFI_P103,VFI_P103,VFI_P113,VFI_P113,VFI_P118,VFI_C115,VFI_C119,VFI_C100,VFI_C96,VFI_C96,VFI_C96,VFI_C96,VFI_C96,VFI_C96,VFI_C96
ing_total,P102_01,P103_01,P103_01,P113_01,P113_01,P118,C115,C119,C100,C96,C96,C96,C96,C96,C96,C96
ing_corrientes,P102_02,P103_02,P103_02,P113_02,P113_02,P118_1,C115_1,C119_1,C100_1,C96_1,C96_1,C96_1,C96_1,C96_1,C96_1,C96_1
ing_capital,P102_60,P103_33,P103_33,P113_33,P113_33,P118_2,C115_2,C119_2,C100_2,C96_2,C96_2,C96_2,C96_2,C96_2,C96_2,C96_2
ing_transferencias,—,P103_36,P103_36,P113_36,P113_36,P118_3,C115_3,C119_3,C100_3,C96_3,C96_3,C96_3,C96_3,C96_3,C96_3,C96_3
ing_financiamiento,P102_71 + P102_74,P103_43,P103_43,P113_43,P113_43,P118_4,C115_4,C119_4,C100_4,C96_4,C96_4,C96_4,C96_4,C96_4,C96_4,C96_4
gas_informo,VFI_P104,VFI_P104,VFI_P104,VFI_P105,VFI_P114,VFI_P119,VFI_C116,VFI_C120,VFI_C101,VFI_C97,VFI_C97,VFI_C97,VFI_C97,VFI_C97,VFI_C97,VFI_C97
gas_total,P104_01,P104_01,P104_01,P105_01,P114_01,P119,C116,C120,C101,C97,C97,C97,C97,C97,C97,C97
gas_corrientes,P104_02,P104_02,P104_02,P105_02,P114_02,P119_1,C116_1,C120_1,C101_1,C97_1,C97_1,C97_1,C97_1,C97_1,C97_1,C97_1


### 3.2 Espacios sobrantes

Se quitan los espacios al inicio y al final, y los espacios dobles dentro del texto quedan en uno.

In [8]:
t = aplicar(crudo, "3.2 espacios", {c: F.trim(F.regexp_replace(F.col(c), r"\s+", " ")) for c in texto}, CAMBIOS)

3.2 espacios: valores cambiados por edición y columna


,tipo,ing_informo,ing_total,ing_corrientes,ing_capital,ing_transferencias,ing_financiamiento,ing_financiamiento__suma1,gas_informo,gas_total,gas_corrientes,gas_capital,gas_servicio_deuda
edicion,,,,,,,,,,,,,
2009,0,0,16,16,16,0,16,16,0,16,16,16,16
2010,0,0,11,11,11,11,11,0,0,11,11,11,11
2011,0,0,6,6,6,6,6,0,0,6,6,6,6
2013,0,0,0,3,0,0,0,0,0,0,0,0,0
2015,2,0,0,0,0,0,0,0,0,0,0,0,0
2019,0,0,1,1,1,1,1,0,0,0,0,0,0
2023,0,1,1,1,1,1,1,0,1,1,1,1,1


Ejemplos (antes → después):


,edicion,columna,antes,después
0,2015,tipo,« »,«»
1,2023,ing_informo,« »,«»


### 3.3 Vacíos a nulos

Un texto vacío (`""`) pasa a `null`. Un monto vacío queda **nulo, no 0**, para distinguir "no hay dato" de "cero".

In [9]:
t = aplicar(t, "3.3 vacíos a nulos", {c: F.when(F.col(c) != "", F.col(c)) for c in texto}, CAMBIOS)

3.3 vacíos a nulos: valores cambiados por edición y columna


,tipo,ing_informo,ing_total,ing_corrientes,ing_capital,ing_transferencias,ing_financiamiento,ing_financiamiento__suma1,gas_informo,gas_total,gas_corrientes,gas_capital,gas_servicio_deuda
edicion,,,,,,,,,,,,,
2009,0,0,16,16,16,0,16,16,0,16,16,16,16
2010,0,0,11,11,11,11,11,0,0,11,11,11,11
2011,0,0,6,6,6,6,6,0,0,6,6,6,6
2013,0,0,0,3,0,0,0,0,0,0,0,0,0
2015,2,0,0,0,0,0,0,0,0,0,0,0,0
2019,0,0,1,1,1,1,1,0,0,0,0,0,0
2023,0,1,1,1,1,1,1,0,1,1,1,1,1


Ejemplos (antes → después):


,edicion,columna,antes,después
0,2015,tipo,«»,None
1,2023,ing_informo,«»,None


### 3.4 Filas vacías

Una fila es vacía si no tiene ningún dato: tipo, ingresos y gastos todos nulos. Una fila con departamento `00` también se trata como vacía.

**Aquí solo se marcan** con `_fila_vacia`. La regla de calidad decide qué hacer con ellas (parte 5).

In [10]:
t = t.withColumn("_fila_vacia", reduce(lambda a, b: a & b, [F.col(c).isNull() for c in datos]) |
                                 F.col("ccdd").isNull() | F.col("ccdd").isin("0", "00")).localCheckpoint()
v = a_pandas(t.groupBy("edicion", "archivo").agg(contar(F.col("_fila_vacia")).alias("filas_vacias")).where("filas_vacias > 0"))
print("Filas vacías marcadas:", int(v["filas_vacias"].sum()) if len(v) else 0)
if len(v):
    display(v)
    display(a_pandas(t.where("_fila_vacia").select("edicion", "archivo", "_fila_origen", *CODIGOS, "tipo").limit(5)))

Filas vacías marcadas: 0


### 3.5 Códigos con ceros a la izquierda y `ubigeo`

- **Ceros a la izquierda:** `ccdd`, `ccpp` y `ccdi` se completan a 2 dígitos (`1` → `01`).
- **`ubigeo`:** = `ccdd` + `ccpp` + `ccdi`. Si algún código no es numérico, queda nulo.
- **Conteo:** `filas_con_ubigeo` cuenta filas. Hasta 2017 ingresos y gastos vienen en dos archivos, así que cada municipalidad aparece dos veces; `municipalidades` cuenta los `ubigeo` distintos. Se juntan en una fila en el paso 3.9.
- **Duplicados:** se cuenta cuántas veces aparece cada `ubigeo` en su archivo (`_n_ubigeo_archivo`), para la regla de duplicados.

In [11]:
t = aplicar(t, "3.5 ceros a la izquierda", {c: F.when(F.col(c).rlike(r"^\d{1,2}$"), F.lpad(F.col(c), 2, "0")).otherwise(F.col(c)) for c in CODIGOS}, CAMBIOS)
t = (t.withColumn("ubigeo", F.when(reduce(lambda a, b: a & b, [F.col(c).rlike(r"^\d{2}$") for c in CODIGOS]), F.concat(*[F.col(c) for c in CODIGOS])))
      .withColumn("_n_ubigeo_archivo", F.count("*").over(Window.partitionBy("edicion", "archivo", "ubigeo")))
      .localCheckpoint())
display(a_pandas(t.groupBy("edicion").agg(F.count("ubigeo").alias("filas_con_ubigeo"), F.countDistinct("ubigeo").alias("municipalidades"), contar(F.col("ubigeo").isNull()).alias("ubigeo_nulo"),
                                          contar(F.col("_n_ubigeo_archivo") > 1).alias("ubigeo_repetido")).orderBy("edicion")).set_index("edicion").T)

3.5 ceros a la izquierda: valores cambiados por edición y columna


,ccdd,ccpp,ccdi
edicion,,,
2015,0,0,1
2023,920,0,0
2024,3,3,4


Ejemplos (antes → después):


,edicion,columna,antes,después
0,2023,ccdd,«1»,«01»
1,2024,ccpp,«5»,«05»


edicion,2009,2010,2011,2012,2013,2015,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
filas_con_ubigeo,3668,3668,3676,3676,3676,3706,3742,1874,1874,1874,1874,1874,1891,1891,1891,1892
municipalidades,1834,1834,1838,1838,1838,1853,1871,1874,1874,1874,1874,1874,1891,1891,1891,1892
ubigeo_nulo,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
ubigeo_repetido,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 3.6 Tipo de municipalidad

Según el catálogo del INEI: `1` → PROVINCIAL, `2` → DISTRITAL, `3` → CENTRO POBLADO (fuera de alcance).
Se guarda el código original en `tipo_codigo`, y un valor fuera del catálogo queda nulo en `tipo_municipalidad`.

In [12]:
TIPOS = ID["tipo_valores"]
etiqueta = reduce(lambda acc, kv: acc.when(F.col("tipo_codigo") == kv[0], F.lit(kv[1])), TIPOS.items(), F.when(F.lit(False), F.lit(None)))
t = t.withColumnRenamed("tipo", "tipo_codigo").withColumn("tipo_municipalidad", etiqueta).localCheckpoint()
x = a_pandas(t.groupBy("edicion", "archivo", "tipo_codigo", "tipo_municipalidad").count())
x["tipo_municipalidad"] = x["tipo_municipalidad"].fillna("FUERA DEL CATÁLOGO o VACÍO")
display(x.pivot_table(index=["edicion", "archivo"], columns="tipo_municipalidad", values="count", aggfunc="sum", fill_value=0))

tipo_municipalidad                        DISTRITAL  FUERA DEL CATÁLOGO o VACÍO  PROVINCIAL
edicion archivo                                                                            
2009    C094.csv                               1639                           0         195
        C095.csv                               1639                           0         195
2010    C094.csv                               1639                           0         195
        C095.csv                               1639                           0         195
2011    C094.csv                               1643                           0         195
        C095.csv                               1643                           0         195
2012    C104.csv                               1643                           0         195
        C105.csv                               1643                           0         195
2013    C104.csv                               1643                           0         195
        C105.csv                               1643                           0         195
2015    C111.csv                               1657                           0         196
        C112.csv                               1656                           2         195
2017    C115.csv                               1675                           0         196
        C116.csv                               1675                           0         196
2018    C06.csv                                1678                           0         196
2019    C04.csv                                1678                           0         196
2020    C04.csv                                1678                           0         196
2021    Base_Renamu_2021_vf.csv                1678                           0         196
2022    Base_RENAMU_2022_f.csv                 1678                           0         196
2023    Base-Datos_2023_f.csv                  1695                           0         196
2024    Base-Datos_2024_f.csv                  1695                           0         196
2025    Base-Datos_2025_f_.csv                 1695                           0         196
2026    Base_de_Datos_F01_RENAMU2026.csv       1696                           0         196

### 3.7 Montos y banderas a número

- **Montos:** `,` → `.` y se convierten a `double`. Ninguna edición usa separador de miles.
- **Banderas** (`*_informo`): se convierten a entero.
- Si un valor no vacío **no se puede convertir**, queda nulo. Cuántos son por fila queda en `_no_convertidos`, para la regla de montos.
- **"Valores cambiados"** cuenta los montos cuyo texto era distinto del número: los que usaban coma decimal. Las ediciones con punto o enteros (2017, 2023, 2024) no cambian de valor, solo de tipo.

In [13]:
montos_todos = [c for c in t.columns if c.startswith(("ing_", "gas_")) and not c.endswith("_informo")]
no_conv = reduce(lambda a, b: a + b, [(F.col(c).isNotNull() & F.regexp_replace(F.col(c), ",", ".").cast("double").isNull()).cast("int") for c in montos_todos])
t = t.withColumn("_no_convertidos", no_conv)
t = aplicar(t, "3.7 montos a número", {**{c: F.regexp_replace(F.col(c), ",", ".").cast("double") for c in montos_todos},
                                        **{c: F.col(c).cast("int") for c in BANDERAS}}, CAMBIOS, ejemplos=2)
print("Valores no convertidos (quedan nulos):", int(a_pandas(t.agg(F.sum("_no_convertidos").alias("n")))["n"][0]))

3.7 montos a número: valores cambiados por edición y columna


,ing_total,ing_corrientes,ing_capital,ing_transferencias,ing_financiamiento,ing_financiamiento__suma1,gas_total,gas_corrientes,gas_capital,gas_servicio_deuda
edicion,,,,,,,,,,
2009,1801,1801,370,0,24,1412,1807,1801,1806,760
2010,1801,1758,120,1801,1461,0,1807,1803,1805,586
2011,1816,1764,104,1817,1252,0,1810,1807,1804,316
2012,1814,1758,102,1815,1345,0,1820,1822,1810,318
2013,1826,1787,97,1821,1421,0,1824,1823,1814,398
2015,1841,1823,73,1837,1451,0,1831,1834,1808,159
2018,1860,1838,77,1851,1514,0,1857,1855,1827,257
2019,1847,1853,66,1851,1575,0,1844,1852,1825,193
2020,1852,1856,68,1839,1510,0,1860,1860,1824,138


Ejemplos (antes → después):


,edicion,columna,antes,después
0,2009,ing_total,"«475632,27»","«475,632.27»"
1,2009,ing_corrientes,"«340230,59»","«340,230.59»"


Valores no convertidos (quedan nulos): 0


### 3.8 Columnas derivadas

- **`anio_dato`:** el año al que corresponde el dato, según el mapeo (2009–2015 = edición; desde 2017, edición − 1).
- **Suma de 2009 (financiamiento):** en 2009 el cuadro de ingresos separa la línea IV en dos columnas:

| Código | Qué trae |
| --- | --- |
| `P102_71` | IV. Financiamiento, pero solo el endeudamiento |
| `P102_74` | Saldo de balance (lo que sobró del año anterior), fuera de la IV |

Desde 2010 el saldo de balance ya viene dentro de la IV. Para que 2009 sea comparable: `ing_financiamiento` = `P102_71` + `P102_74`.
Sin esta suma el total de 2009 no cuadra con I + II + IV (visto en `05a`). Cambian solo las filas con saldo de balance distinto de 0.

In [14]:
anio = reduce(lambda acc, kv: acc.when(F.col("edicion") == kv[0], F.lit(kv[1]["anio_dato"])), ED.items(), F.when(F.lit(False), F.lit(None)))
t = t.withColumn("anio_dato", anio.cast("int")).localCheckpoint()
display(a_pandas(t.groupBy("edicion", "anio_dato").agg(F.count("*").alias("filas")).orderBy("edicion")).set_index("edicion").T)

# variables que el mapeo arma sumando columnas: {columna: (edición, [códigos])}
LISTAS = {pref + v: (a, c) for a, e in ED.items() for b, pref in (("ingresos", "ing_"), ("gastos", "gas_"))
          for v, c in e[b].items() if isinstance(c, list)}
sumas = {}
for c in [c for c in t.columns if "__suma" in c]:
    sumas.setdefault(c.split("__suma")[0], []).append(c)
for base, extras in sumas.items():
    a, cods = LISTAS[base]
    nuevo = F.when(reduce(lambda x, y: x | y, [F.col(x).isNotNull() for x in [base] + extras]),
                   reduce(lambda x, y: x + y, [F.coalesce(F.col(x), F.lit(0.0)) for x in [base] + extras]))
    print(f"Ejemplos {a}: {base} = {' + '.join(cods)}")
    display(a_pandas(t.where((F.col("edicion") == a) & (F.col(extras[0]) != 0)).select(
        "edicion", "ubigeo", F.format_number(F.col(base), 2).alias(f"{cods[0]} (antes)"),
        *[F.format_number(F.col(x), 2).alias(k) for x, k in zip(extras, cods[1:])],
        F.format_number(nuevo, 2).alias(f"{base} (después)")).limit(3)))
    t = aplicar(t, f"3.8 suma {base}", {base: nuevo}, CAMBIOS, ejemplos=0)
    filas = t.where((F.col("edicion") == a) & F.col("bloques").contains("ingresos" if base.startswith("ing_") else "gastos")).count()
    print(f"{CAMBIOS[-1]['valores_cambiados']} de {filas} filas de {a} tienen {cods[1]} distinto de 0 y cambian; en las demás el valor queda igual.")
    t = t.drop(*extras)

edicion,2009,2010,2011,2012,2013,2015,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
anio_dato,2009,2010,2011,2012,2013,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
filas,3668,3668,3676,3676,3676,3706,3742,1874,1874,1874,1874,1874,1891,1891,1891,1892


Ejemplos 2009: ing_financiamiento = P102_71 + P102_74


,edicion,ubigeo,P102_71 (antes),P102_74,ing_financiamiento (después)
0,2009,010102,0.00,"40,689.68","40,689.68"
1,2009,010105,0.00,"165,760.00","165,760.00"
2,2009,010110,0.00,"50,229.58","50,229.58"


3.8 suma ing_financiamiento: valores cambiados por edición y columna


,ing_financiamiento
edicion,
2009,1791


1791 de 1834 filas de 2009 tienen P102_74 distinto de 0 y cambian; en las demás el valor queda igual.


### 3.9 Unión de ingresos y gastos

- **Archivos distintos (2009–2017):** se unen por `edicion` + `ubigeo`. La identificación (códigos y tipo) sale del archivo de ingresos; si falta, del de gastos.
- **Mismo archivo (desde 2018):** ya están juntos.
- **Filas sin `ubigeo`:** no se pueden unir, así que pasan como filas sueltas para que las vea la regla de calidad.
- **Trazabilidad:** cada fila conserva el número de fila de origen de cada archivo.

In [15]:
IDENT = ["edicion", "anio_dato", "ubigeo", *CODIGOS, "tipo_codigo", "tipo_municipalidad"]
TRAZA = ["_fila_origen", "_archivo_origen", "_sha256_origen", "_fecha_descarga", "_fila_vacia", "_n_ubigeo_archivo", "_no_convertidos"]
uno = t.where(F.col("bloques") == "ingresos,gastos").select(*IDENT, *ING, *GAS,
          *[F.col(x).alias(x + "_ing") for x in TRAZA], *[F.col(x).alias(x + "_gas") for x in TRAZA])
ti = t.where(F.col("bloques") == "ingresos").select(*IDENT, *ING, *[F.col(x).alias(x + "_ing") for x in TRAZA])
tg = t.where(F.col("bloques") == "gastos").select("edicion", "ubigeo", *[F.col(c).alias(c + "_g") for c in IDENT if c not in ("edicion", "ubigeo")],
                                                    *GAS, *[F.col(x).alias(x + "_gas") for x in TRAZA])
con_ub = (ti.where(F.col("ubigeo").isNotNull()).join(tg.where(F.col("ubigeo").isNotNull()), ["edicion", "ubigeo"], "full"))
for c in IDENT:
    if c not in ("edicion", "ubigeo"):
        con_ub = con_ub.withColumn(c, F.coalesce(F.col(c), F.col(c + "_g"))).drop(c + "_g")
sin_ub = ti.where(F.col("ubigeo").isNull()).unionByName(tg.where(F.col("ubigeo").isNull())
          .select("edicion", "ubigeo", *[F.col(c + "_g").alias(c) for c in IDENT if c not in ("edicion", "ubigeo")], *GAS,
                  *[F.col(x + "_gas") for x in TRAZA]), allowMissingColumns=True)
tabla = uno.unionByName(con_ub, allowMissingColumns=True).unionByName(sin_ub, allowMissingColumns=True).localCheckpoint()

antes = a_pandas(t.groupBy("edicion").agg(F.count("*").alias("filas_en_archivos")))
despues = a_pandas(tabla.groupBy("edicion").agg(F.count("*").alias("filas_unidas"),
          contar(F.col("_fila_origen_ing").isNull()).alias("solo_en_gastos"), contar(F.col("_fila_origen_gas").isNull()).alias("solo_en_ingresos")))
display(antes.merge(despues, on="edicion").sort_values("edicion").set_index("edicion"))

,filas_en_archivos,filas_unidas,solo_en_gastos,solo_en_ingresos
edicion,,,,
2009,3668,1834,0,0
2010,3668,1834,0,0
2011,3676,1838,0,0
2012,3676,1838,0,0
2013,3676,1838,0,0
2015,3706,1853,0,0
2017,3742,1871,0,0
2018,1874,1874,0,0
2019,1874,1874,0,0


### 3.10 Estados de cada bloque

| Estado | Significado |
| --- | --- |
| `INFORMO` | La bandera de informante es 1 |
| `NO_INFORMO` | La bandera es 0 |
| `SIN_RESPUESTA` | La municipalidad está en el archivo, pero la bandera viene vacía |
| `SIN_CUADRO` | La municipalidad no está en el archivo de ese bloque |

In [16]:
def estado(pref):
    return (F.when(F.col(f"_fila_origen_{pref}").isNull(), F.lit("SIN_CUADRO"))
             .when(F.col(f"{pref}_informo").isNull(), F.lit("SIN_RESPUESTA"))
             .when(F.col(f"{pref}_informo") == 1, F.lit("INFORMO"))
             .otherwise(F.lit("NO_INFORMO")))
tabla = (tabla.withColumn("estado_ingresos", estado("ing")).withColumn("estado_gastos", estado("gas"))).localCheckpoint()
e = a_pandas(tabla.groupBy("edicion", "estado_ingresos", "estado_gastos").count())
display(e.pivot_table(index="edicion", columns=["estado_ingresos", "estado_gastos"], values="count", aggfunc="sum", fill_value=0))

estado_ingresos,INFORMO,NO_INFORMO,SIN_RESPUESTA
estado_gastos,INFORMO,NO_INFORMO,SIN_RESPUESTA
edicion,,,
2009,1818,16,0
2010,1823,11,0
2011,1832,6,0
2012,1838,0,0
2013,1838,0,0
2015,1853,0,0
2017,1871,0,0
2018,1874,0,0


## 4. Comparación antes / después

- **Filas:** las de la carga cruda (paso 1) frente a las de la tabla limpia, por edición.
- **Montos:** la suma de ingresos y gastos totales tras convertirlos (paso 3.7) frente a la suma tras la unión (paso 3.9). **Deben ser iguales:** limpiar y unir no puede cambiar los montos.

In [17]:
f_ini = foto_inicial.groupby("edicion")["count"].sum().rename("filas_carga_cruda")
m_antes = a_pandas(t.groupBy("edicion").agg(F.round(F.sum("ing_total") / 1e6, 2).alias("ingresos_antes"),
                                            F.round(F.sum("gas_total") / 1e6, 2).alias("gastos_antes"))).set_index("edicion")
m_desp = a_pandas(tabla.groupBy("edicion").agg(F.count("*").alias("filas_tabla"),
                  contar(F.coalesce(F.col("_fila_vacia_ing"), F.col("_fila_vacia_gas"))).alias("filas_vacias"),
                  contar(F.col("ubigeo").isNull()).alias("sin_ubigeo"),
                  F.round(F.sum("ing_total") / 1e6, 2).alias("ingresos_despues"),
                  F.round(F.sum("gas_total") / 1e6, 2).alias("gastos_despues"))).set_index("edicion")
comp = pd.concat([f_ini, m_desp[["filas_tabla", "filas_vacias", "sin_ubigeo"]], m_antes, m_desp[["ingresos_despues", "gastos_despues"]]], axis=1).sort_index()
comp["montos_iguales"] = (comp["ingresos_antes"] == comp["ingresos_despues"]) & (comp["gastos_antes"] == comp["gastos_despues"])
display(comp)
assert comp["montos_iguales"].all(), "La limpieza o la unión cambiaron los montos"
print("Montos en millones de soles. 'filas_carga_cruda' suma ambos archivos cuando ingresos y gastos vienen separados (2009–2017).")

,filas_carga_cruda,filas_tabla,filas_vacias,sin_ubigeo,ingresos_antes,gastos_antes,ingresos_despues,gastos_despues,montos_iguales
edicion,,,,,,,,,
2009,3668,1834,0,0,18412.71,15815.59,18412.71,15815.59,True
2010,3668,1834,0,0,20087.06,17720.41,20087.06,17720.41,True
2011,3676,1838,0,0,20850.23,16906.22,20850.23,16906.22,True
2012,3676,1838,0,0,26274.78,22345.16,26274.78,22345.16,True
2013,3676,1838,0,0,26564.55,24789.55,26564.55,24789.55,True
2015,3706,1853,0,0,23394.60,21436.75,23394.60,21436.75,True
2017,3742,1871,0,0,25926.79,23243.07,25926.79,23243.07,True
2018,1874,1874,0,0,28791.40,24818.12,28791.40,24818.12,True
2019,1874,1874,0,0,32387.13,27374.13,32387.13,27374.13,True


Montos en millones de soles. 'filas_carga_cruda' suma ambos archivos cuando ingresos y gastos vienen separados (2009–2017).


### Registro de cambios (todos los pasos)

In [18]:
reg = pd.DataFrame(CAMBIOS)
display(reg.groupby("paso")["valores_cambiados"].agg(columnas="count", valores_cambiados="sum"))

,columnas,valores_cambiados
paso,,
3.2 espacios,16,318
3.3 vacíos a nulos,16,318
3.5 ceros a la izquierda,3,931
3.7 montos a número,12,173941
3.8 suma ing_financiamiento,1,1791


## 5. Reglas de calidad

Las reglas están en `config.yaml`. Se aplican sobre la tabla limpia (una fila por municipalidad y edición), en este orden:

| # | Regla | Qué verifica | Acción si falla |
| --- | --- | --- | --- |
| 1 | `SLV-F3-003` | `ubigeo` de 6 dígitos (sin departamento `00`) y fila con datos | Cuarentena |
| 2 | `SLV-F3-001` | Cada municipalidad aparece una sola vez por edición | Cuarentena (todas las copias) |
| 3 | `SLV-F3-002` | Tipo 1 (provincial) o 2 (distrital). El 3 (centro poblado) es **fuera de alcance**; otro valor o vacío, cuarentena | Cuarentena o fuera de alcance |
| 4 | `SLV-TODAS-001` | Privacidad: la tabla final solo tiene columnas permitidas | Detener |
| 5 | `SLV-TODAS-002` | Cuadre: filas de la carga cruda = Silver + cuarentena + fuera de alcance, por archivo; y los montos | Detener |
| 6 | `SLV-F3-005` | El total es la suma de sus categorías (1 sol de tolerancia por categoría) | Registrar |
| 7 | `SLV-F3-006` | Montos numéricos y no negativos | Registrar |
| 8 | `SLV-F3-007` | Bandera de informante: no informó (0) o vacía (sin respuesta) | Registrar |

- **Cuarentena:** la fila no entra a Silver; se guarda aparte con la regla y el motivo. Si falla varias, se anota la primera en este orden.
- **Registrar:** la fila sí entra a Silver; el id de la regla queda en `_alertas` para revisarla en Gold (página de auditoría).
- **Detener:** el notebook se detiene y no escribe nada.
- Las reglas que cruzan fuentes (`SLV-F3-004` tipo RENAMU vs. SIAF y `SLV-X-001` municipalidades del SIAF sin RENAMU) se aplican cuando esté el Silver del SIAF.

Todas se ejecutan en la celda siguiente, que muestra un solo resumen. Las de tipo *detener* paran el notebook **después** de mostrarlo. El detalle del cuadre está en 5.1.

In [19]:
import re
VERIF = []   # resultado de cada regla (va a la bitácora)
def resultado(id_, evaluadas, fallidas, nota=""):
    r = REGLAS[id_]
    estado = "OK" if fallidas == 0 else {"cuarentena": "CUARENTENA", "registrar": "ALERTA", "detener": "DETENIDO"}[r["accion"]]
    VERIF.append({"regla": id_, "descripcion": r["descripcion"], "severidad": r["severidad"], "accion": r["accion"],
                  "filas_evaluadas": int(evaluadas), "filas_fallidas": int(fallidas), "estado": estado, "nota": nota})

f = lambda c: F.coalesce(c, F.lit(False))
N_TABLA = tabla.count()

# 1–3. Críticas: cada fila recibe su destino (silver, cuarentena con _regla y _motivo, o fuera_de_alcance)
falla = {
    "SLV-F3-003": (F.col("ubigeo").isNull() | ~F.col("ubigeo").rlike(r"^\d{6}$") | F.col("ubigeo").startswith("00")
                   | f(F.col("_fila_vacia_ing")) | f(F.col("_fila_vacia_gas"))),
    "SLV-F3-001": f(F.col("_n_ubigeo_archivo_ing") > 1) | f(F.col("_n_ubigeo_archivo_gas") > 1),
    "SLV-F3-002": ~f(F.col("tipo_codigo").isin("1", "2", "3")),
}
motivo = {"SLV-F3-003": "ubigeo inválido o fila vacía", "SLV-F3-001": "ubigeo repetido en su archivo",
          "SLV-F3-002": "tipo de municipalidad vacío o fuera del catálogo"}
regla_col = reduce(lambda acc, kv: acc.when(kv[1], F.lit(kv[0])), falla.items(), F.when(F.lit(False), F.lit(None)))
motivo_col = reduce(lambda acc, kv: acc.when(F.col("_regla") == kv[0], F.lit(kv[1])), motivo.items(), F.when(F.lit(False), F.lit(None)))
tabla = (tabla.withColumn("_regla", regla_col).withColumn("_motivo", motivo_col)
              .withColumn("_destino", F.when(F.col("_regla").isNotNull(), "cuarentena")
                                       .when(F.col("tipo_codigo") == "3", "fuera_de_alcance").otherwise("silver"))
              .localCheckpoint())
n = a_pandas(tabla.agg(*[contar(c).alias(k) for k, c in falla.items()], contar(F.col("tipo_codigo") == "3").alias("tipo_3")))
for k in falla:
    resultado(k, N_TABLA, n[k][0], "centro poblado = fuera de alcance: %d" % n["tipo_3"][0] if k == "SLV-F3-002" else "")
destinos = a_pandas(tabla.groupBy("edicion", "_destino").count())
en_silver = F.col("_destino") == "silver"

# 4. Privacidad: lista fija de columnas de salida, ninguna puede ser un dato personal
TRAZA_OUT = [f"{x}_{s}" for s in ("ing", "gas") for x in ("_archivo_origen", "_sha256_origen", "_fecha_descarga", "_fila_origen")]
COLUMNAS_SILVER = ["edicion", "anio_dato", "ubigeo", "tipo_codigo", "tipo_municipalidad", *ING, *GAS,
                   "estado_ingresos", "estado_gastos", "_alertas", "_run_id", "_fecha_proceso_utc", "_fuente", *TRAZA_OUT]
COLUMNAS_CUARENTENA = [c for c in COLUMNAS_SILVER if c != "_alertas"] + ["_regla", "_motivo"]
SENSIBLES = re.compile(r"correo|e_?mail|telef|celular|fax|dni|documento|alcalde|nombre|apellido|direcc|domicilio|usuario|web", re.I)
sensibles = [c for c in set(COLUMNAS_SILVER + COLUMNAS_CUARENTENA) if SENSIBLES.search(c)]
resultado("SLV-TODAS-001", len(set(COLUMNAS_SILVER + COLUMNAS_CUARENTENA)), len(sensibles), "columnas revisadas")

# 5. Cuadre: filas por archivo de origen y montos por edición (detalle en 5.1)
lado = lambda s: tabla.select("edicion", F.col(f"_archivo_origen_{s}").alias("_archivo_origen"), F.col(f"_fila_origen_{s}").alias("fila"), "_destino")
destino_filas = lado("ing").unionByName(lado("gas")).where(F.col("fila").isNotNull()).dropDuplicates(["edicion", "_archivo_origen", "fila"])
cuadre = (a_pandas(crudo.groupBy("edicion", "_archivo_origen").agg(F.count("*").alias("carga_cruda")))
          .merge(a_pandas(destino_filas.groupBy("edicion", "_archivo_origen").pivot("_destino", ["silver", "cuarentena", "fuera_de_alcance"]).count()).fillna(0),
                 on=["edicion", "_archivo_origen"], how="outer").fillna(0))
cuadre["archivo"] = cuadre["_archivo_origen"].str.split("/").str[-1]
cuadre["cuadra"] = cuadre["carga_cruda"] == cuadre["silver"] + cuadre["cuarentena"] + cuadre["fuera_de_alcance"]
m_dest = a_pandas(tabla.groupBy("edicion").agg(F.round(F.sum("ing_total") / 1e6, 2).alias("ingresos_destinos"),
                                               F.round(F.sum("gas_total") / 1e6, 2).alias("gastos_destinos"))).set_index("edicion")
cm = m_antes.join(m_dest).sort_index()
cm["cuadra"] = (cm["ingresos_antes"] == cm["ingresos_destinos"]) & (cm["gastos_antes"] == cm["gastos_destinos"])
resultado("SLV-TODAS-002", int(cuadre["carga_cruda"].sum()), int((~cuadre["cuadra"]).sum() + (~cm["cuadra"]).sum()), "archivos y ediciones que no cuadran")

# 6. El total es la suma de sus categorías (1 sol de tolerancia por categoría), solo filas de Silver con total
CAT = {"ing": [c for c in ING if c not in ("ing_informo", "ing_total")], "gas": [c for c in GAS if c not in ("gas_informo", "gas_total")]}
for p_, cats in CAT.items():
    tabla = (tabla.withColumn(f"_dif_{p_}", F.col(f"{p_}_total") - reduce(lambda a, b: a + b, [F.coalesce(F.col(c), F.lit(0.0)) for c in cats]))
                  .withColumn(f"_f005_{p_}", f(en_silver & (F.abs(F.col(f"_dif_{p_}")) > reduce(lambda a, b: a + b, [F.col(c).isNotNull().cast("int") for c in cats])))))
tabla = tabla.withColumn("_f005", F.col("_f005_ing") | F.col("_f005_gas"))

# 7. Montos numéricos y no negativos (los negativos se conservan y se revisan en Gold)
MONTOS_COLS = [c for c in ING + GAS if not c.endswith("_informo")]
neg = reduce(lambda a, b: a | b, [f(F.col(c) < 0) for c in MONTOS_COLS])
noconv = f(F.coalesce(F.col("_no_convertidos_ing"), F.lit(0)) + F.coalesce(F.col("_no_convertidos_gas"), F.lit(0)) > 0)
tabla = tabla.withColumn("_f006", en_silver & (neg | noconv))

# 8. Bandera de informante: no informó (0) o vacía; se cuenta si aun así traen montos > 0
SIN = ["NO_INFORMO", "SIN_RESPUESTA"]
con_montos = reduce(lambda a, b: a | b, [f(F.col(c) > 0) for p_ in CAT for c in [f"{p_}_total", *CAT[p_]]])
tabla = tabla.withColumn("_f007", en_silver & (F.col("estado_ingresos").isin(SIN) | F.col("estado_gastos").isin(SIN))).localCheckpoint()

r = a_pandas(tabla.agg(
    contar(en_silver & (F.col("ing_total").isNotNull() | F.col("gas_total").isNotNull())).alias("eval_005"), contar(F.col("_f005")).alias("f005"),
    contar(en_silver).alias("silver"), contar(F.col("_f006")).alias("f006"), contar(en_silver & noconv).alias("noconv"),
    contar(F.col("_f006") & neg).alias("negativos"), contar(F.col("_f007")).alias("f007"), contar(F.col("_f007") & con_montos).alias("f007_montos"))).iloc[0]
resultado("SLV-F3-005", r["eval_005"], r["f005"])
resultado("SLV-F3-006", r["silver"], r["f006"], f"negativos: {r['negativos']} · no convertidos: {r['noconv']}")
resultado("SLV-F3-007", r["silver"], r["f007"], f"con montos > 0: {r['f007_montos']}")

display(pd.DataFrame(VERIF)[["regla", "accion", "filas_evaluadas", "filas_fallidas", "estado", "nota"]].set_index("regla"))
if sensibles:
    raise RuntimeError(f"SLV-TODAS-001: columnas sensibles en la salida: {sensibles}")
if not (cuadre["cuadra"].all() and cm["cuadra"].all()):
    raise RuntimeError("SLV-TODAS-002: el cuadre no coincide; no se escribe nada (ver 5.1)")

,accion,filas_evaluadas,filas_fallidas,estado,nota
regla,,,,,
SLV-F3-003,cuarentena,29841,0,OK,
SLV-F3-001,cuarentena,29841,0,OK,
SLV-F3-002,cuarentena,29841,0,OK,centro poblado = fuera de alcance: 0
SLV-TODAS-001,detener,32,0,OK,columnas revisadas
SLV-TODAS-002,detener,42747,0,OK,archivos y ediciones que no cuadran
SLV-F3-005,registrar,29807,1,ALERTA,
SLV-F3-006,registrar,29841,8,ALERTA,negativos: 8 · no convertidos: 0
SLV-F3-007,registrar,29841,34,ALERTA,con montos > 0: 0


### 5.1 Cuadre (`SLV-TODAS-002`)

- **Filas, por archivo:** cada fila de la carga cruda (parte 1) termina en Silver, en cuarentena o fuera de alcance. Se cuenta con el número de fila de origen de cada archivo, así que vale aunque 2009–2017 junte dos archivos en una fila.
- **Montos, por edición:** la suma de los totales tras la limpieza es igual a la suma tras convertirlos (paso 3.7).

In [20]:
display(cuadre.drop(columns="_archivo_origen").astype({c: int for c in ["carga_cruda", "silver", "cuarentena", "fuera_de_alcance"]})
              .set_index(["edicion", "archivo"]).sort_index())
display(cm)

carga_cruda  silver  cuarentena  fuera_de_alcance  cuadra
edicion archivo                                                                                    
2009    C094.csv                                 1834    1834           0                 0    True
        C095.csv                                 1834    1834           0                 0    True
2010    C094.csv                                 1834    1834           0                 0    True
        C095.csv                                 1834    1834           0                 0    True
2011    C094.csv                                 1838    1838           0                 0    True
        C095.csv                                 1838    1838           0                 0    True
2012    C104.csv                                 1838    1838           0                 0    True
        C105.csv                                 1838    1838           0                 0    True
2013    C104.csv                                 1838    1838           0                 0    True
        C105.csv                                 1838    1838           0                 0    True
2015    C111.csv                                 1853    1853           0                 0    True
        C112.csv                                 1853    1853           0                 0    True
2017    C115.csv                                 1871    1871           0                 0    True
        C116.csv                                 1871    1871           0                 0    True
2018    C06.csv                                  1874    1874           0                 0    True
2019    C04.csv                                  1874    1874           0                 0    True
2020    C04.csv                                  1874    1874           0                 0    True
2021    Base_Renamu_2021_vf.csv                  1874    1874           0                 0    True
2022    Base_RENAMU_2022_f.csv                   1874    1874           0                 0    True
2023    Base-Datos_2023_f.csv                    1891    1891           0                 0    True
2024    Base-Datos_2024_f.csv                    1891    1891           0                 0    True
2025    Base-Datos_2025_f_.csv                   1891    1891           0                 0    True
2026    Base_de_Datos_F01_RENAMU2026.csv         1892    1892           0                 0    True

,ingresos_antes,gastos_antes,ingresos_destinos,gastos_destinos,cuadra
edicion,,,,,
2009,18412.71,15815.59,18412.71,15815.59,True
2010,20087.06,17720.41,20087.06,17720.41,True
2011,20850.23,16906.22,20850.23,16906.22,True
2012,26274.78,22345.16,26274.78,22345.16,True
2013,26564.55,24789.55,26564.55,24789.55,True
2015,23394.60,21436.75,23394.60,21436.75,True
2017,25926.79,23243.07,25926.79,23243.07,True
2018,28791.40,24818.12,28791.40,24818.12,True
2019,32387.13,27374.13,32387.13,27374.13,True


## 6. Escritura en Parquet

- **Silver:** `data/silver/renamu_finanzas/`, particionado por `edicion`. Se sobrescribe en cada ejecución.
- **Cuarentena:** `data/quarantine/renamu_finanzas/`, con `_regla` y `_motivo` (se escribe aunque esté vacía, para que el esquema exista).
- **Fuera de alcance:** solo se cuenta.
- **Trazabilidad:** `_run_id`, `_fecha_proceso_utc`, `_fuente` y, por archivo de origen (`_ing` / `_gas`), el archivo, su `sha256`, la fecha de descarga y el número de fila.
- Al final se vuelve a leer lo escrito y se compara con lo esperado.

In [21]:
fecha_proceso = datetime.now(timezone.utc)
alertas = F.concat_ws(",", *[F.when(F.col(c), F.lit(r)) for c, r in (("_f005", "SLV-F3-005"), ("_f006", "SLV-F3-006"), ("_f007", "SLV-F3-007"))])
final = (tabla.withColumn("_alertas", F.when(alertas != "", alertas))
              .withColumn("_run_id", F.lit(RUN_ID)).withColumn("_fecha_proceso_utc", F.lit(fecha_proceso)).withColumn("_fuente", F.lit(MAPEO["fuente"])))
silver_df = final.where(F.col("_destino") == "silver").select(*COLUMNAS_SILVER)
cuar_df = final.where(F.col("_destino") == "cuarentena").select(*COLUMNAS_CUARENTENA)

DESTINO = ruta("silver") / MAPEO["tabla"]
DESTINO_Q = ruta("quarantine") / MAPEO["tabla"]
silver_df.write.mode("overwrite").partitionBy("edicion").parquet(str(DESTINO))
cuar_df.write.mode("overwrite").parquet(str(DESTINO_Q))

esperado = {d: int(destinos.loc[destinos["_destino"] == d, "count"].sum()) for d in ("silver", "cuarentena", "fuera_de_alcance")}
leido = spark.read.parquet(str(DESTINO))
n_leido, n_q = leido.count(), spark.read.parquet(str(DESTINO_Q)).count()
assert n_leido == esperado["silver"] and n_q == esperado["cuarentena"], "Lo escrito no coincide con lo esperado"
print(f"Silver: {n_leido} filas en {DESTINO.relative_to(RAIZ)} · Cuarentena: {n_q} filas en {DESTINO_Q.relative_to(RAIZ)} · "
      f"Fuera de alcance: {esperado['fuera_de_alcance']}")
leido.printSchema()

Silver: 29841 filas en data/silver/renamu_finanzas · Cuarentena: 0 filas en data/quarantine/renamu_finanzas · Fuera de alcance: 0
root
 |-- anio_dato: integer (nullable = true)
 |-- ubigeo: string (nullable = true)
 |-- tipo_codigo: string (nullable = true)
 |-- tipo_municipalidad: string (nullable = true)
 |-- ing_informo: integer (nullable = true)
 |-- ing_total: double (nullable = true)
 |-- ing_corrientes: double (nullable = true)
 |-- ing_capital: double (nullable = true)
 |-- ing_transferencias: double (nullable = true)
 |-- ing_financiamiento: double (nullable = true)
 |-- gas_informo: integer (nullable = true)
 |-- gas_total: double (nullable = true)
 |-- gas_corrientes: double (nullable = true)
 |-- gas_capital: double (nullable = true)
 |-- gas_servicio_deuda: double (nullable = true)
 |-- estado_ingresos: string (nullable = true)
 |-- estado_gastos: string (nullable = true)
 |-- _alertas: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _fecha_proceso_utc

## 7. Bitácora y auditoría

Se agrega una línea a `data/silver/bitacora_ejecuciones.jsonl` con el bloque `volumenes`, que tiene el mismo formato que en Bronze (capa, fuente, tabla, archivos, filas, bytes, fecha de los datos). También guarda el resultado de cada regla y los cambios de la limpieza. Luego se reconstruye `data/auditoria/volumenes.parquet`.

In [22]:
from src import auditoria
fin = datetime.now(timezone.utc)
cambios = pd.DataFrame(CAMBIOS).groupby("paso", sort=False)["valores_cambiados"].sum()
n_arch, n_bytes = auditoria.tamano_en_disco(DESTINO)
fecha_datos = a_pandas(crudo.agg(F.max("_fecha_descarga").alias("f")))["f"][0]   # descarga de Bronze de la que vienen los datos
vol = auditoria.volumen("silver", "F3", MAPEO["tabla"], filas=esperado["silver"], archivos=n_arch, bytes=n_bytes, fecha_datos=fecha_datos,
                        filas_carga_cruda=int(foto_inicial["count"].sum()), archivos_origen=len(foto_inicial),
                        filas_cuarentena=esperado["cuarentena"], filas_fuera_de_alcance=esperado["fuera_de_alcance"])
entrada = {"run_id": RUN_ID, "notebook": "05b_silver_renamu", "capa": "silver", "inicio_utc": inicio.isoformat(), "fin_utc": fin.isoformat(),
           "tabla": MAPEO["tabla"], "ediciones": sorted(ED), "volumenes": [vol],
           "destino": str(DESTINO.relative_to(RAIZ)), "destino_cuarentena": str(DESTINO_Q.relative_to(RAIZ)),
           "cambios_limpieza": {k: int(v) for k, v in cambios.items()}, "verificaciones": VERIF}
auditoria.escribir_bitacora("silver", entrada)
print("Bitácora actualizada:", RUN_ID, f"({(fin - inicio).total_seconds():.0f} s)")

df_aud = auditoria.construir_auditoria()
display(auditoria.ultimo_volumen(df_aud)[["capa", "fuente", "tabla", "archivos", "filas", "mb", "fecha_datos", "fin_utc"]])

Bitácora actualizada: silver_renamu_20261122T194755Z_ad7628 (47 s)


,capa,fuente,tabla,archivos,filas,mb,fecha_datos,fin_utc
0,bronze,F1,siaf_ingreso,18,15204055,7336.2,2026-11-22,2026-11-22 17:37:44.000000+00:00
1,bronze,F2,sismepre_predial,7,493843,59.1,2026-11-22,2026-11-22 17:37:44.000000+00:00
2,bronze,F3,renamu,1109,2240471,354.5,2026-11-20,2026-11-22 17:37:44.000000+00:00
3,silver,F3,renamu_finanzas,43,29841,2.3,2026-11-20,2026-11-22 19:48:42.412889+00:00


## 8. Resumen final

**Tabla `renamu_finanzas`:** 29 841 filas, una por municipalidad y edición, en 16 ediciones (2009–2026, sin 2014 ni 2016). Está en `data/silver/renamu_finanzas/`, particionada por `edicion`.

- **Recorrido de las filas:** 42 747 filas de 23 archivos → 29 841 tras unir ingresos y gastos (hasta 2017 vienen en dos archivos) → **0 en cuarentena y 0 fuera de alcance** → 29 841 en Silver.
- **Cuadre:** cada fila de origen termina en un destino, y los montos por edición no cambian con la limpieza.
- **Limpieza:** espacios (318 valores), vacíos a nulos (318), ceros a la izquierda (931), montos a número (173 941) y la suma del financiamiento de 2009 (1 791).
- **Alertas** (las filas entran a Silver, con el id de la regla en `_alertas`):
  - `SLV-F3-005`, 1 fila: en 2026, ubigeo 150113, el total de ingresos no es igual a la suma de sus categorías (diferencia de S/ 11 709).
  - `SLV-F3-006`, 8 filas con montos negativos: 5 en el financiamiento de 2009–2018 y 3 en los ingresos corrientes de 2025.
  - `SLV-F3-007`, 34 filas: 33 municipalidades que no informaron en 2009–2011 y 1 sin respuesta en 2023. Ninguna trae montos.
- **Columnas:**
  - llaves: `edicion`, `anio_dato`, `ubigeo`, `tipo_codigo`, `tipo_municipalidad`;
  - ingresos: `ing_*` (6);
  - gastos: `gas_*` (5);
  - estados: `estado_ingresos`, `estado_gastos`;
  - control: `_alertas`.
- **Trazabilidad:** `_run_id`, `_fecha_proceso_utc` y `_fuente`. Además, **por archivo de origen**, con sufijo `_ing` / `_gas` (hasta 2017 ingresos y gastos vienen en CSV distintos): `_archivo_origen`, `_sha256_origen`, `_fecha_descarga` y `_fila_origen`.
- **Auditoría:** la bitácora registra Silver F3 con 29 841 filas, 2,3 MB y datos de la descarga del 2026-11-20. Aparece en `data/auditoria/volumenes.parquet` debajo de Bronze F3.

In [23]:
spark.stop()